## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [44]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [45]:
google_api_key = os.getenv("GOOGLE_API_KEY")

In [61]:
load_dotenv(override=True)
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [62]:
reader = PdfReader("me/Profile (1).pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [63]:
print(linkedin)

   
Contact
amithvardhan729@gmail.com
www.linkedin.com/in/amith-var
(LinkedIn)
Top Skills
Data Wrangling
Exploratory Data Analysis
Machine Learning
Certifications
The Data Science Course: Complete
Data Science Bootcamp
IBM Data Analyst
IBM Data Science
Google Data Analytics
Google Advanced Data Analytics
Amith Reddy
Data Scientist
United States
Summary
Data Scientist with a Master's in Information Systems Technologies
and expertise in Python, SQL, and machine learning. Skilled in
data analytics, data visualization, predictive modeling, natural
language processing, deep learning, and generative AI. Experienced
in fraud detection, credit risk modeling, recommendation systems,
and multilingual NLP solutions. Proficient in tools and frameworks
including TensorFlow, scikit-learn, Tableau, MongoDB, and MySQL.
Strong background in data wrangling, feature engineering, model
deployment, and large language models (GPT-3.5, Llama 3). Hands-
on experience in building end-to-end AI and data science

In [64]:
with open("me/summary_1.txt", "r", encoding="utf-8") as f:
    summary = f.read()
print(summary)

Hi! I am Amith. I did my masters in Information Systems Technologies from Wilmington University. Right now I am a Data Scientist volunteer at DUTA, which is a news aggregator startup based in India. I am working on building an AI agent that can summarize news articles and provide insights on the same. 


In [65]:
name_1 = "Ed Donner"
name = "Amith"

In [66]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say no."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

print(system_prompt)


You are acting as Amith. You are answering questions on Amith's website, particularly questions related to Amith's career, background, skills and experience. Your responsibility is to represent Amith for interactions on the website as faithfully as possible. You are given a summary of Amith's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say no.

## Summary:
Hi! I am Amith. I did my masters in Information Systems Technologies from Wilmington University. Right now I am a Data Scientist volunteer at DUTA, which is a news aggregator startup based in India. I am working on building an AI agent that can summarize news articles and provide insights on the same. 

## LinkedIn Profile:
   
Contact
amithvardhan729@gmail.com
www.linkedin.com/in/amith-var
(LinkedIn)
Top Skills
Data Wrangling
Exploratory Data Analysis
Machine Learn

In [67]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [68]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [69]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [70]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

print(evaluator_system_prompt)


You are an evaluator that decides whether a response to a question is acceptable. You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. The Agent is playing the role of Amith and is representing Amith on their website. The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. The Agent has been provided with context on Amith in the form of their summary and LinkedIn details. Here's the information:

## Summary:
Hi! I am Amith. I did my masters in Information Systems Technologies from Wilmington University. Right now I am a Data Scientist volunteer at DUTA, which is a news aggregator startup based in India. I am working on building an AI agent that can summarize news articles and provide insights on the same. 

## LinkedIn Profile:
   
Contact
amithvardhan729@gmail.com
www.linkedin.com/in/amith-var
(LinkedI

In [71]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [72]:
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [ ]:
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = ollama_client.beta.chat.completions.parse(model="llama3.2", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed # Returns a structured output

In [74]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
reply = response.choices[0].message.content

In [75]:
print(reply)

That's an interesting question! Based on my current background and work experience, I do not hold a patent. My focus has been on developing and implementing AI and data science solutions, particularly in areas like news summarization, translation, and building recommendation systems.


In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1]) # Structured output

Evaluation(is_acceptable=False, feedback='The response is direct and to the point, but could benefit from a more professional tone. For example, instead of saying \'I do not hold a patent\', one could phrase it as "At present, I do not have any patents to my credit" or "My work experience does not include holding any patents". Adding these phrases would make the response sound more polished and representative of Amith\'s professional demeanor.')

In [85]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
    return response.choices[0].message.content

In [86]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [87]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply
Failed evaluation - retrying
The response does not provide a clear or helpful answer to the user's question. The Agent's attempt at mimicking regional accents and colloquialisms ('O-nay', 'I do-nay ot-nay') in this context comes across as trying too hard to be relatable and ends up looking unprofessional. A more accurate and straightforward response, such as 'No, I don't hold any patents' would have been more suitable.
